Data Collection I (API)
API: MusicBrainz (https://musicbrainz.org/doc/MusicBrainz_API) — free, no API key, no signup.

Summary
- Endpoints used: /ws/2/artist/ (search by name) and /ws/2/release/ (browse by artist).
- For every top-10 finisher in every Eurovision grand final from 2015 to 2025 (100 artist-year rows, 99 unique artists, 2020 skipped — contest cancelled).
- For each artist we search for the full release discography to measure release frequency before vs after their Eurovision year.
- Expected output: 1000+ release records across ~99 artists.
- Limitations: a few very recent artists (e.g. 2025 finalists) may not yet have MusicBrainz entries — the notebook prints which ones failed.
- Authentication: MusicBrainz needs no API key, only a descriptive User-Agent. If we were using an authenticated API (Spotify, Last.fm) we would store the key in a .env file (see markdown cell below).


A note on .env

MusicBrainz is no-auth, so we do not need an API key. For an authenticated API the pattern is:

```
# .env  (never committed)
LASTFM_API_KEY=abcd1234
```
```python
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.environ["LASTFM_API_KEY"]
```

.env is in .gitignore so it never reaches GitHub.


In [1]:
!pip install -q pyarrow

In [2]:
import requests
import time
import json
import pandas as pd
from datetime import datetime

HEADERS = {
    "User-Agent": "AlbertSchool-DataProject/1.0 (marta.prandin@student.example.com)",
    "Accept": "application/json",
}
BASE = "https://musicbrainz.org/ws/2"

The 100 Eurovision top-10 finishers, 2015–2025:

Source: Wikipedia year pages (referenced statically here; actual scraping is in 02_scraping.ipynb file).

Each row is (artist, year, song, country, rank) — rank 1 to 10 within each year's grand final.

In [3]:
TOP10 = [
    ("Måns Zelmerlöw", 2015, "Heroes", "Sweden", 1),
    ("Polina Gagarina", 2015, "A Million Voices", "Russia", 2),
    ("Il Volo", 2015, "Grande amore", "Italy", 3),
    ("Loïc Nottet", 2015, "Rhythm Inside", "Belgium", 4),
    ("Aminata Savadogo", 2015, "Love Injected", "Latvia", 5),
    ("Guy Sebastian", 2015, "Tonight Again", "Australia", 6),
    ("Elina Born and Stig Rästa", 2015, "Goodbye to Yesterday", "Estonia", 7),
    ("Mørland", 2015, "A Monster Like Me", "Norway", 8),
    ("Bojana Stamenov", 2015, "Beauty Never Lies", "Serbia", 9),
    ("Nadav Guedj", 2015, "Golden Boy", "Israel", 10),
    ("Jamala", 2016, "1944", "Ukraine", 1),
    ("Dami Im", 2016, "Sound of Silence", "Australia", 2),
    ("Sergey Lazarev", 2016, "You Are the Only One", "Russia", 3),
    ("Poli Genova", 2016, "If Love Was a Crime", "Bulgaria", 4),
    ("Frans", 2016, "If I Were Sorry", "Sweden", 5),
    ("Amir", 2016, "J'ai cherché", "France", 6),
    ("Iveta Mukuchyan", 2016, "LoveWave", "Armenia", 7),
    ("Michał Szpak", 2016, "Color of Your Life", "Poland", 8),
    ("Donny Montell", 2016, "I've Been Waiting for This Night", "Lithuania", 9),
    ("Laura Tesoro", 2016, "What's the Pressure", "Belgium", 10),
    ("Salvador Sobral", 2017, "Amar pelos dois", "Portugal", 1),
    ("Kristian Kostov", 2017, "Beautiful Mess", "Bulgaria", 2),
    ("SunStroke Project", 2017, "Hey, Mamma!", "Moldova", 3),
    ("Blanche", 2017, "City Lights", "Belgium", 4),
    ("Robin Bengtsson", 2017, "I Can't Go On", "Sweden", 5),
    ("Francesco Gabbani", 2017, "Occidentali's Karma", "Italy", 6),
    ("Ilinca and Alex Florea", 2017, "Yodel It!", "Romania", 7),
    ("Joci Pápai", 2017, "Origo", "Hungary", 8),
    ("Jacques Houdek", 2017, "My Friend", "Croatia", 9),
    ("JOWST", 2017, "Grab the Moment", "Norway", 10),
    ("Netta", 2018, "Toy", "Israel", 1),
    ("Eleni Foureira", 2018, "Fuego", "Cyprus", 2),
    ("Cesár Sampson", 2018, "Nobody but You", "Austria", 3),
    ("Michael Schulte", 2018, "You Let Me Walk Alone", "Germany", 4),
    ("Ermal Meta and Fabrizio Moro", 2018, "Non mi avete fatto niente", "Italy", 5),
    ("Mikolas Josef", 2018, "Lie to Me", "Czech Republic", 6),
    ("Benjamin Ingrosso", 2018, "Dance You Off", "Sweden", 7),
    ("Elina Nechayeva", 2018, "La forza", "Estonia", 8),
    ("Rasmussen", 2018, "Higher Ground", "Denmark", 9),
    ("DoReDoS", 2018, "My Lucky Day", "Moldova", 10),
    ("Duncan Laurence", 2019, "Arcade", "Netherlands", 1),
    ("Mahmood", 2019, "Soldi", "Italy", 2),
    ("Sergey Lazarev", 2019, "Scream", "Russia", 3),
    ("Luca Hänni", 2019, "She Got Me", "Switzerland", 4),
    ("KEiiNO", 2019, "Spirit in the Sky", "Norway", 5),
    ("John Lundvik", 2019, "Too Late for Love", "Sweden", 6),
    ("Chingiz", 2019, "Truth", "Azerbaijan", 7),
    ("Lake Malawi", 2019, "Friend of a Friend", "Czech Republic", 8),
    ("Kate Miller-Heidke", 2019, "Zero Gravity", "Australia", 9),
    ("Hatari", 2019, "Hatrið mun sigra", "Iceland", 10),
    ("Måneskin", 2021, "Zitti e buoni", "Italy", 1),
    ("Barbara Pravi", 2021, "Voilà", "France", 2),
    ("Gjon's Tears", 2021, "Tout l'univers", "Switzerland", 3),
    ("Daði og Gagnamagnið", 2021, "10 Years", "Iceland", 4),
    ("Go_A", 2021, "Shum", "Ukraine", 5),
    ("Blind Channel", 2021, "Dark Side", "Finland", 6),
    ("Destiny", 2021, "Je Me Casse", "Malta", 7),
    ("The Roop", 2021, "Discoteque", "Lithuania", 8),
    ("Manizha", 2021, "Russian Woman", "Russia", 9),
    ("The Black Mamba", 2021, "Love Is on My Side", "Portugal", 10),
    ("Kalush Orchestra", 2022, "Stefania", "Ukraine", 1),
    ("Sam Ryder", 2022, "Space Man", "United Kingdom", 2),
    ("Chanel", 2022, "SloMo", "Spain", 3),
    ("Cornelia Jakobs", 2022, "Hold Me Closer", "Sweden", 4),
    ("Konstrakta", 2022, "In corpore sano", "Serbia", 5),
    ("Mahmood and Blanco", 2022, "Brividi", "Italy", 6),
    ("Zdob și Zdub and Frații Advahov", 2022, "Trenulețul", "Moldova", 7),
    ("Amanda Tenfjord", 2022, "Die Together", "Greece", 8),
    ("MARO", 2022, "Saudade, saudade", "Portugal", 9),
    ("Subwoolfer", 2022, "Give That Wolf a Banana", "Norway", 10),
    ("Loreen", 2023, "Tattoo", "Sweden", 1),
    ("Käärijä", 2023, "Cha Cha Cha", "Finland", 2),
    ("Noa Kirel", 2023, "Unicorn", "Israel", 3),
    ("Marco Mengoni", 2023, "Due vite", "Italy", 4),
    ("Alessandra", 2023, "Queen of Kings", "Norway", 5),
    ("Tvorchi", 2023, "Heart of Steel", "Ukraine", 6),
    ("Gustaph", 2023, "Because of You", "Belgium", 7),
    ("Alika", 2023, "Bridges", "Estonia", 8),
    ("Monika Linkytė", 2023, "Stay", "Lithuania", 9),
    ("Voyager", 2023, "Promise", "Australia", 10),
    ("Nemo", 2024, "The Code", "Switzerland", 1),
    ("Baby Lasagna", 2024, "Rim Tim Tagi Dim", "Croatia", 2),
    ("Alyona Alyona and Jerry Heil", 2024, "Teresa & Maria", "Ukraine", 3),
    ("Slimane", 2024, "Mon amour", "France", 4),
    ("Eden Golan", 2024, "Hurricane", "Israel", 5),
    ("Bambie Thug", 2024, "Doomsday Blue", "Ireland", 6),
    ("Angelina Mango", 2024, "La noia", "Italy", 7),
    ("Ladaniva", 2024, "Jako", "Armenia", 8),
    ("Marcus and Martinus", 2024, "Unforgettable", "Sweden", 9),
    ("Iolanda", 2024, "Grito", "Portugal", 10),
    ("JJ", 2025, "Wasted Love", "Austria", 1),
    ("Yuval Raphael", 2025, "New Day Will Rise", "Israel", 2),
    ("Tommy Cash", 2025, "Espresso Macchiato", "Estonia", 3),
    ("KAJ", 2025, "Bara bada bastu", "Sweden", 4),
    ("Lucio Corsi", 2025, "Volevo essere un duro", "Italy", 5),
    ("Klavdia", 2025, "Asteromata", "Greece", 6),
    ("Louane", 2025, "Maman", "France", 7),
    ("Shkodra Elektronike", 2025, "Zjerm", "Albania", 8),
    ("Ziferblat", 2025, "Bird of Pray", "Ukraine", 9),
    ("Zoë Më", 2025, "Voyage", "Switzerland", 10),
]

Two helper functions:

- search_artist_mbid: search MusicBrainz by artist name and it returns the first matching MBID (MusicBrainz unique ID).
- get_artist_releases: browse all releases (singles, EPs, albums) for a given MBID, with pagination.

Both follow MusicBrainz's 1-request-per-second policy and handle the 429 "rate limited" response.

In [4]:
def search_artist_mbid(name):
    response = requests.get(
        f"{BASE}/artist/",
        params={"query": f'artist:"{name}"', "fmt": "json", "limit": 1},
        headers=HEADERS,
        timeout=10,
    )
    response.raise_for_status()
    data = response.json()
    if data.get("artists"):
        return data["artists"][0]["id"]
    return None


def get_artist_releases(mbid):
    all_releases = []
    offset = 0
    while True:
        response = requests.get(
            f"{BASE}/release/",
            params={"artist": mbid, "fmt": "json", "limit": 100, "offset": offset},
            headers=HEADERS,
            timeout=10,
        )
        if response.status_code == 429:
            wait = int(response.headers.get("Retry-After", 2))
            print(f"  rate limited, waiting {wait}s...")
            time.sleep(wait)
            continue
        response.raise_for_status()
        chunk = response.json().get("releases", [])
        all_releases.extend(chunk)
        if len(chunk) < 100:
            break
        offset += 100
        time.sleep(1)
    return all_releases

Run the collection:

100 artists × (1 search + 1 release browse) =  around 200 requests. With a 1-second polite delay, this takes about 3-4 minutes

In [5]:
records = []
not_found = []
mbid_cache = {}

for artist_name, year, song, country, rank in TOP10:
    print(f"{rank:>2}. {artist_name} ({year})")

    if artist_name in mbid_cache:
        mbid = mbid_cache[artist_name]
    else:
        try:
            mbid = search_artist_mbid(artist_name)
        except requests.RequestException as e:
            print(f"     ! search error: {e}")
            continue
        mbid_cache[artist_name] = mbid
        time.sleep(1)

    if mbid is None:
        print("     ! not found in MusicBrainz")
        not_found.append((artist_name, year))
        continue

    try:
        releases = get_artist_releases(mbid)
    except requests.RequestException as e:
        print(f"     ! release fetch error: {e}")
        continue
    print(f"     -> {len(releases)} releases")

    for rel in releases:
        records.append({
            "artist": artist_name,
            "eurovision_year": year,
            "eurovision_song": song,
            "eurovision_country": country,
            "eurovision_rank": rank,
            "artist_mbid": mbid,
            "release_id": rel.get("id"),
            "release_title": rel.get("title"),
            "release_date": rel.get("date"),
            "release_country": rel.get("country"),
            "release_status": rel.get("status"),
        })

    time.sleep(1)

print(f"\nDone. {len(records)} release records collected.")
if not_found:
    print(f"\nArtists not found in MusicBrainz ({len(not_found)}):")
    for a, y in not_found:
        print(f"  - {a} ({y})")

 1. Måns Zelmerlöw (2015)
     -> 45 releases
 2. Polina Gagarina (2015)
     ! not found in MusicBrainz
 3. Il Volo (2015)
     -> 57 releases
 4. Loïc Nottet (2015)
     -> 10 releases
 5. Aminata Savadogo (2015)
     ! not found in MusicBrainz
 6. Guy Sebastian (2015)
     -> 73 releases
 7. Elina Born and Stig Rästa (2015)
     ! not found in MusicBrainz
 8. Mørland (2015)
     -> 2 releases
 9. Bojana Stamenov (2015)
     -> 1 releases
10. Nadav Guedj (2015)
     -> 6 releases
 1. Jamala (2016)
     -> 41 releases
 2. Dami Im (2016)
     -> 36 releases
 3. Sergey Lazarev (2016)
     ! not found in MusicBrainz
 4. Poli Genova (2016)
     ! not found in MusicBrainz
 5. Frans (2016)
     -> 163 releases
 6. Amir (2016)
     -> 38 releases
 7. Iveta Mukuchyan (2016)
     -> 3 releases
 8. Michał Szpak (2016)
     -> 4 releases
 9. Donny Montell (2016)
     -> 22 releases
10. Laura Tesoro (2016)
     -> 10 releases
 1. Salvador Sobral (2017)
     -> 13 releases
 2. Kristian Kostov (201

Save raw JSON checkpoint:
so that we never re-hit MusicBrainz for the same run.

In [6]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
raw_json_name = f"musicbrainz_raw_{stamp}.json"

dump = {
    "queried_at": stamp,
    "endpoint_base": BASE,
    "top10_input": TOP10,
    "artists_not_found": not_found,
    "records": records,
}
with open(raw_json_name, "w") as f:
    json.dump(dump, f, indent=2)
print("Saved raw JSON:", raw_json_name)

Saved raw JSON: musicbrainz_raw_20260520_092643.json


Build the DataFrame and inspect it:

In [7]:
df = pd.DataFrame(records)
print("Shape:", df.shape)
print()
df.info()
print()
df.head()

Shape: (2793, 11)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2793 entries, 0 to 2792
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   artist              2793 non-null   object
 1   eurovision_year     2793 non-null   int64 
 2   eurovision_song     2793 non-null   object
 3   eurovision_country  2793 non-null   object
 4   eurovision_rank     2793 non-null   int64 
 5   artist_mbid         2793 non-null   object
 6   release_id          2793 non-null   object
 7   release_title       2793 non-null   object
 8   release_date        2758 non-null   object
 9   release_country     2367 non-null   object
 10  release_status      2738 non-null   object
dtypes: int64(2), object(9)
memory usage: 240.2+ KB



,artist,eurovision_year,eurovision_song,eurovision_country,eurovision_rank,artist_mbid,release_id,release_title,release_date,release_country,release_status
0,Måns Zelmerlöw,2015,Heroes,Sweden,1,9a21832f-5104-42fa-b881-17a1ca66bfdb,0631a490-ea76-475d-9d4d-8be2e2937526,Hope & Glory,2009-03-04,SE,Official
1,Måns Zelmerlöw,2015,Heroes,Sweden,1,9a21832f-5104-42fa-b881-17a1ca66bfdb,1708903d-70f6-48d8-ae59-2052aaf8d132,Barcelona Sessions,2014-02-05,SE,Official
2,Måns Zelmerlöw,2015,Heroes,Sweden,1,9a21832f-5104-42fa-b881-17a1ca66bfdb,1c04b970-5961-4a42-a234-25ea372515fd,Love Love Peace Peace,2016-05-05,XW,Official
3,Måns Zelmerlöw,2015,Heroes,Sweden,1,9a21832f-5104-42fa-b881-17a1ca66bfdb,1e4f01ae-e213-476b-8f1c-6b5398dabc7a,Circles and Squares,2021-02-19,XW,Official
4,Måns Zelmerlöw,2015,Heroes,Sweden,1,9a21832f-5104-42fa-b881-17a1ca66bfdb,23969704-cf1b-402f-80e8-2f8a63c8229c,Beautiful Life,2013-09-13,None,Official


Save the tabular dataset:

In [8]:
csv_name = f"eurovision_artists_discography_{stamp}.csv"
df.to_csv(csv_name, index=False)
print("Saved:", csv_name)

Saved: eurovision_artists_discography_20260520_092643.csv


Download both files:

In [10]:
from google.colab import files
files.download(raw_json_name)
files.download(csv_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>